In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/archive (5).zip"  # Your mounted Drive file path

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/data")  # Extract into Colab local disk

print("✅ Extracted files:", os.listdir("/content/data"))

In [ ]:
import pandas as pd

df = pd.read_csv("data/indiana_reports.csv")
print("✅ Loaded:", df.shape)

# step1: load data + Preprocessing


- **Text Cleaning**:
  - Fill missing values in `impression`, `findings`, `indication`, and `comparison` fields with placeholders like "No impression".
  - Define a cleaning function to:
    - Remove HTML tags.
    - Expand English contractions (e.g., won't → will not).
    - Remove bullet numbers, placeholders (XXXXX), and URLs.
    - Remove non-letter characters (keep periods for sentence separation).
    - Normalize spaces and lowercase all text.
  - Apply the cleaning function to all four text fields.

- **Create Captions**:
  - Concatenate `impression` and `findings` for each report.
  - Remove redundant (duplicate) sentences in the concatenated text.
  - Save the result in a new column `caption`.

- **Word Cloud Visualization**:
  - Generate and display word clouds for:
    - `indication` field.
    - `findings` field.
    - Combined `caption` field.
  - Helps visualize the most frequent words in each text field.

- **Match Image Files**:
  - Match each report's `uid` to its corresponding image file in the `images_normalized` folder.
  - The image filename must start with the UID and end with `.dcm.png`.
  - Save the matched image path to a new column `image_path`.

- **Save Final Dataset**:
  - Filter the dataset to keep only rows where a matching image was found.
  - Save a new CSV file `data.csv` containing:
    - `image_path`: Path to the corresponding X-ray image.
    - `caption`: Cleaned and merged report text.



In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from bs4 import BeautifulSoup
from tqdm import tqdm
import numpy as np
import re


# -----------------------------
# Load Dataset
# -----------------------------
df = pd.read_csv("data/indiana_reports.csv")


# -----------------------------
# Text Cleaning
# -----------------------------

# Fill Missing Text
df["impression"] = df["impression"].fillna("No impression")
df["findings"] = df["findings"].fillna("No findings")
df["indication"] = df["indication"].fillna("No indication")
df["comparison"] = df["comparison"].fillna("No comparison")

# Cleaning function
def decontracted(phrase):
    phrase = re.sub(r"won\'t", "will not", phrase)
    phrase = re.sub(r"can\'t", "can not", phrase)
    phrase = re.sub(r"n\'t", " not", phrase)
    phrase = re.sub(r"\'re", " are", phrase)
    phrase = re.sub(r"\'s", " is", phrase)
    phrase = re.sub(r"\'d", " would", phrase)
    phrase = re.sub(r"\'ll", " will", phrase)
    phrase = re.sub(r"\'t", " not", phrase)
    phrase = re.sub(r"\'ve", " have", phrase)
    phrase = re.sub(r"\'m", " am", phrase)
    return phrase

def clean_text_column(text_series):
    cleaned = []
    for text in tqdm(text_series.fillna(""), desc="Cleaning Text"):
        text = BeautifulSoup(text, 'lxml').get_text() # Remove HTML tags
        text = decontracted(text)                 # Expand contractions
        text = re.sub(r"\d+\.", "", text)         # remove bullet numbers like 1. 2.
        text = re.sub(r"X+", "", text)            # Remove placeholders like XXXXX
        text = re.sub(r"http\S+", "", text)       # remove URLs
        text = re.sub(r"[^a-zA-Z. ]", " ", text)  # remove non-letter chars (keep dots)
        text = re.sub(r"\s+", " ", text)          # normalize spaces
        text = text.lower().strip()               # # Lowercase and remove extra spaces
        cleaned.append(text if text else np.nan)
    return cleaned

#Apply Text Cleaning
df["impression"] = clean_text_column(df["impression"])
df["findings"] = clean_text_column(df["findings"])
df["indication"] = clean_text_column(df["indication"])
df["comparison"] = clean_text_column(df["comparison"])

# -----------------------------
# Create Captions (Impression + Findings)
# -----------------------------

def remove_redundancy(text):
    # Split on any period followed by optional whitespace
    sentences = re.split(r'\.\s*', text)
    # Clean and filter empty parts
    sentences = [s.strip() for s in sentences if s.strip()]
    # Remove exact duplicates while preserving order
    unique_sentences = list(dict.fromkeys(sentences))
    # Rejoin with ". " and add final period
    return '. '.join(unique_sentences) + '.'


df["caption"] = (
    df["indication"].fillna('') + ". " +
    df["impression"].fillna('') + ". " +
    df["findings"].fillna('')
).apply(remove_redundancy)

# -----------------------------
# Word Clouds
# -----------------------------
text_indication = " ".join(df["indication"].dropna())
wordcloud_ind = WordCloud(width=800, height=300, background_color="white").generate(text_indication)
plt.figure(figsize=(10, 4))
plt.imshow(wordcloud_ind, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud – Indication")
plt.show()

text_findings = " ".join(df["findings"].dropna())
wordcloud_find = WordCloud(width=800, height=300, background_color="white").generate(text_findings)
plt.figure(figsize=(10, 4))
plt.imshow(wordcloud_find, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud – Findings")
plt.show()

# New Word Cloud for Caption
text_caption = " ".join(df["caption"].dropna())
wordcloud_caption = WordCloud(width=800, height=300, background_color="white").generate(text_caption)
plt.figure(figsize=(10, 4))
plt.imshow(wordcloud_caption, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud – Caption (Impression + Findings)")
plt.show()


# -----------------------------------------------
# Match Image Files and save the data set
# -----------------------------------------------

# This block links each X-ray report to its image file and saves a clean dataset of image-caption pairs in data.csv.
image_folder = "data/images/images_normalized"
available_files = set(os.listdir(image_folder)) #ists all filenames in the folder
                                                #Convert to set:For fast lookup — checking membership in a set is
                                                  #O(1) vs O(n) for a list

def match_uid_to_filename(uid):
    prefix = str(uid)
    for fname in available_files:
        if fname.startswith(prefix) and fname.endswith(".dcm.png"):
            return os.path.join(image_folder, fname)
    return None

df["image_path"] = df["uid"].apply(match_uid_to_filename)


# Save
df_filtered = df[df["image_path"].notnull()]
df_filtered[["image_path", "caption"]].to_csv("data.csv", index=False)
print(f"✅ Final data saved: {len(df_filtered)} image-caption pairs")



# Step 2: Visual Feature Extraction and Fusion (ViT + Handcrafted Features)

**1. Load Pretrained Vision Transformer (ViT) Model**:
  - `vit-base-patch16-224-in21k` pretrained on ImageNet-21k.
  - Only the ViT encoder is used for feature extraction (no classification head).
  - ViT Architecture Overview:
    - **224×224**: Input image size.
    - **16×16**: Patch size.
    - **14×14**: Number of patches → 196 patches.
    - **[CLS] token**: Special learnable token added at the beginning.
    - **Transformer Encoder**: Processes [CLS] token + patch tokens using self-attention layers.
    - **Output**: [CLS] embedding — 768-dimensional vector summarizing the entire image.
  - ✅ **The [CLS] token acts as a compact global summary of the whole image.**

**2. Define PyTorch Dataset Class**:
  - Custom `ChestXrayDataset` class to:
    - Load X-ray images and associated captions.
    - Apply ViT preprocessing (resize to 224×224, normalize pixel values).

**3. Create DataLoader**:
  - Wrap the dataset into a PyTorch `DataLoader`.
  - Load images in batches (batch size = 4) for efficient processing.



**4. Extract ViT Features**:
  - Pass preprocessed images through the ViT model.
  - Extract the **[CLS] token** embeddings for each image:
    - 768-dimensional feature vector summarizing the whole image.

**5. Feature Fusion**:
  - Concatenate each image’s ViT feature vector (768-D) with its handcrafted features (2-D).
  - Final feature vector for each image: **770 dimensions** (768 + 2).






In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import ViTModel, AutoImageProcessor  # HuggingFace Vision Transformer (ViT) model and preprocessor
from PIL import Image                          # For opening images
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler
import cv2
import numpy as np

# ==============================
# Set Device : Use GPU if available; otherwise, use CPU
# ==============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ==============================
# Load Pretrained ViT Model
# ==============================
'''
#"google/vit-base-patch16-224-in21k"
ViT-Base: "base" refers to the model size (medium-sized: 86 million parameters).
Patch16: Images are split into 16×16 pixel patches.
224: The model expects input images resized to 224×224 pixels.
in21k: Pretrained on ImageNet-21k — a huge dataset with 21,000 classes (better feature learning).
'''
model_name = "google/vit-base-patch16-224-in21k"   # Pretrained ViT model on ImageNet-21k
vit_model = ViTModel.from_pretrained(model_name).to(device) #Load the ViT encoder (no classification head — just the transformer layers
processor = AutoImageProcessor.from_pretrained(model_name)    # Load preprocessing pipeline (resizing, normalization)
                                                              #ensures your input images are compatible with the pretrained model.

# ==============================
# Load CSV File
# ==============================
df = pd.read_csv("data.csv")

# ==============================
# Dataset Class for Chest X-rays
# ==============================
'''
This class loads an X-ray image and its caption, preprocesses the image for ViT, and returns them together as a dictionary.
{
  'pixel_values': tensor([3, 224, 224]),            # Preprocessed image tensor
  'caption': "no signs of pneumonia. heart size normal.",  # Text caption
  'image_path': "./images/images_normalized/12345.dcm.png" # Image file path
}
'''

class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, processor):
        self.data = dataframe.reset_index(drop=True)  # Reset index to be sure it is clean
        self.processor = processor                   # Preprocessing pipeline

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]                     # Get one row (image + caption)
        image_path = row['image_path']                 # Image path
        caption = row['caption']                       # Corresponding caption
        image = Image.open(image_path).convert("RGB")   # Open image and convert to RGB(3 Channel)
        encoding = self.processor(images=image, return_tensors="pt")  # Preprocess image
        encoding = {k: v.squeeze(0) for k, v in encoding.items()}     # Remove batch dim
        encoding["caption"] = caption                  # Add caption
        encoding["image_path"] = image_path            # Add image path
        return encoding

# ==============================
# Dataloader
# ==============================
dataset = ChestXrayDataset(df, processor) #Create an instance of class ChestXrayDataset.
dataloader = DataLoader(dataset, batch_size=4, shuffle=False)   # Wrap the dataset in a PyTorch DataLoader.
                                                                  #batch_size=4: load 4 images at a time (in one batch).

# ==============================
# Handcrafted Feature Extraction Function
# ==============================
def extract_handcrafted_features(image_path, features=["sobel", "brightness"]):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)   # Load image in grayscale
    img = cv2.resize(img, (224, 224))
    feats = []

    if "sobel" in features:
        sobelx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
        sobely = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
        edge_strength = np.mean(np.hypot(sobelx, sobely))
        feats.append(edge_strength)

    if "brightness" in features:
        hist = cv2.calcHist([img], [0], None, [256], [0, 256])
        brightness_mean = np.mean(hist)
        feats.append(brightness_mean)

    return np.array(feats, dtype=np.float32)

# ==============================
# Feature Extraction Pipeline
# ==============================
vit_model.eval()    # Set ViT model to evaluation mode(not training) ensures we don’t update weights —
                       #very important for feature extraction.

all_features = []   # List to store final ViT-only features

# ==============================
# Extract ViT Features (ViT only — handcrafted ignored)
# ==============================
with torch.no_grad():   # No gradients needed for inference , NOT training.just extracting features (doing a forward pass)
    for batch in dataloader:
        pixel_values = batch["pixel_values"].to(device)    # batch["pixel_values"] contains the preprocessed images
        outputs = vit_model(pixel_values=pixel_values)     # Forward pass through ViT
        cls_embeddings = outputs.last_hidden_state[:, 0, :]  # Extract [CLS] token embeddings (global feature vector)

        for i in range(cls_embeddings.size(0)):
            all_features.append(cls_embeddings[i].cpu())  # Save only ViT features (no handcrafted)

# ==============================
# Final Feature Tensor
# ==============================
image_features = torch.stack(all_features, dim=0)    # Stack all ViT features into a single tensor #[num_images, 768]
# === Sanity Check for NaNs ===
if torch.isnan(image_features).any():
    raise ValueError("⚠️ NaNs detected in extracted image features! Check your input images or preprocessing.")
else:
    print("✅ No NaNs in image features — safe to proceed.")



# Step 3: GPT-2 Decoder with Cross-Attention

### 🧠 Full Flow of Step 3: Report Generation Pipeline

1. **Load Pretrained GPT-2 Model with Cross-Attention**
   - Load GPT-2 (small version) with cross-attention enabled.
   - Set `pad_token_id` for proper padding handling.
  
2. **Prepare Dataset**
   - **Project image features** from 770D → 768D.
   - Define a **custom PyTorch Dataset**:
     - Returns tokenized captions (`input_ids`), attention masks, and projected image features.
   - Create a **DataLoader** for batching.

3. **Training Loop**
   - **Split dataset** into train/validation/test (70/15/15).
   - **Training**:
     - Forward pass: feed tokenized captions.
     - Use image features as cross-attention context.
     - Compute language modeling loss.
     - Backpropagate and update model parameters.
   - **Validation**:
     - Same forward pass, but no gradients.
     - Compute validation loss and perplexity.
   - **Plot** training and validation loss/perplexity curves.

4. **Inference / Report Generation**
   - Set model to **evaluation** mode.
   - Loop over **test dataset**:
     - Pass image features to GPT-2 as cross-attention context.
     - Generate a caption using **beam search**.
     - Decode generated token IDs into text.
     - Decode ground-truth captions for comparison.
   - Save results (image path, reference caption, generated caption) into `generated_reports.csv`.

✅ **Outcome**:
> A trained GPT-2 model capable of generating chest X-ray medical reports conditioned on image features.



                           +--------------------------+
                           |    Pretrained ViT + Hand   |
                           |  Image Features (770-D)    |
                           +-------------+-------------+
                                         |
                                         v
                       +-------------------------------+
                       |  Projection Layer (770 → 768)  |
                       +-------------+-----------------+
                                         |
                                         v
                           +--------------------------+
                           |     GPT-2 (w/ Cross-Attn)  |
                           +-------------+-------------+
                                         |
              +--------------------------+--------------------------+
              |                                                     |
    +---------v---------+                                 +---------v---------+
    |  Training Phase    |                                 |  Inference Phase   |
    +--------------------+                                +--------------------+
    |  - Tokenize captions|                                |  - Generate report |
    |  - Feed input_ids    |                               |  - Beam Search     |
    |  - Feed attention_mask|                              |  - Decode captions |
    |  - Feed image_feature|                               |                    |
    |  - Compute Loss     |                                |                    |
    +--------------------+                                +--------------------+
              |                                                     |
              v                                                     v
     +-----------------+                               +------------------------------+
     |  Training Loss &  |                              | Save to `generated_reports.csv`|
     | Perplexity Curves |                              +------------------------------+
     +------------------+


In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, GPT2Config
from peft import get_peft_model, LoraConfig, TaskType
import torch.nn as nn

# Load tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2", padding_side="left")
tokenizer.pad_token = tokenizer.eos_token

# Load GPT-2 config with cross-attention
config = GPT2Config.from_pretrained("gpt2", add_cross_attention=True)
base_model = GPT2LMHeadModel.from_pretrained("gpt2", config=config)
base_model.config.pad_token_id = tokenizer.pad_token_id

# ✅ LoRA Configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["c_attn", "mlp.c_fc", "mlp.c_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# Wrap base model with LoRA adapters
model = get_peft_model(base_model, lora_config)
model = model.to(device)
model.train()

# === Projection Layer for Image Features ===
image_feat_dim = image_features.shape[1]
gpt2_hidden_size = model.config.hidden_size  # 768
projector = nn.Linear(image_feat_dim, gpt2_hidden_size).to(device)

print(f"✅ GPT-2 (with LoRA + cross-attention) loaded. Projecting features {image_feat_dim} → {gpt2_hidden_size}")




# === Step 3.2 — Caption Tokenization + Dataset Preparation ===

from torch.utils.data import Dataset, DataLoader

'''
Check if:
     Number of rows in the DataFrame df (captions) matches Number of extracted image_features.
Important to make sure each image has a caption.
'''
# Sanity check to ensure matching number of captions and image features
assert len(df) == image_features.shape[0], "Mismatch between captions and image features!"

# Project all image features from 770 to 768 dimensions before training
# === Define Projection Layer (even if 768→768) ===
projector = nn.Linear(image_features.shape[1], model.config.hidden_size).to(device)

with torch.no_grad():
    projected_image_features = projector(image_features.to(device)).cpu()


# Define custom dataset for report generation
'''
this class does two main jobs:
 1. Provides Preprocessed Image Features:
    * It loads the projected image features (768-dimensional vectors) for each chest X-ray.
    * These features are used as cross-attention context in GPT-2 — helping the model generate reports based on image information.

 2. Tokenizes Captions:
    * It takes the corresponding medical report caption (text) for each image.
    * It uses a tokenizer (GPT-2 tokenizer) to:
        Turn the text into token IDs.
        Pad or truncate the sequence to a fixed length (max_length).
        Create an attention mask to distinguish real tokens from padding.
The output:
It returns a dictionary with three elements:
  1. the tokenized caption,
  2. an attention mask (distinguishes real tokens (1) from padding (0))to ignore padding,
  3. image_feature: the projected image feature (for cross-attention).

example:
{
  "input_ids": tensor([ 50256, 345,  8123, ..., 50256, 0, 0, 0]),  # (128,)
  "attention_mask": tensor([ 1, 1, 1, ..., 0, 0, 0]),              # (128,)
  "image_feature": tensor([ 0.12, -0.45, ..., 0.33 ])              # (768,)
}
'''
class ReportGenerationDataset(Dataset):
    def __init__(self, dataframe, projected_features, tokenizer, max_length=128):
        self.data = dataframe.reset_index(drop=True)     # Store the DataFrame
        self.image_features = projected_features         # Store projected image features
        self.tokenizer = tokenizer                       # GPT-2 tokenizer
        self.max_length = max_length                     # Max token length for captions

    def __len__(self):
        return len(self.data)  # Number of samples

    def __getitem__(self, idx):
        # Get image feature
        img_feat = self.image_features[idx]  # [768]

        # Tokenize caption
        caption = self.data.loc[idx, "caption"]
        encoding = self.tokenizer(
            caption,
            padding="max_length",    # Pad caption to max length
            truncation=True,         # Truncate if longer than max length
            max_length=self.max_length,
            return_tensors="pt"      # Return PyTorch tensors
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),         # Caption tokens [max_length]
            "attention_mask": encoding["attention_mask"].squeeze(0),  # Attention mask
            "image_feature": img_feat                              # Projected image feature [768]
        }

# Instantiate the dataset and DataLoader
#building the dataset and dataloader pipeline to serve batches of (image features + tokenized captions) for training GPT-2 with cross-attention.
dataset = ReportGenerationDataset(df, projected_image_features, tokenizer)#create an instance of the ReportGenerationDataset class.
                                                                          #Now, dataset knows how to:For any idx, return a dictionary with:
                                                                                 #1. input_ids (tokenized caption).
                                                                                  #2.attention_mask (for ignoring padding).
                                                                                 #3. image_feature (projected image features for cross-attention).


train_loader = DataLoader(dataset, batch_size=4, shuffle=True)#Create a PyTorch DataLoader,will load batches of data during training.

print(f"✅ Dataset ready: {len(dataset)} samples")



In [ ]:
# === Step 3.3: Full Training Loop with Scheduler and Early Stopping ===

import torch
import math
import time
import matplotlib.pyplot as plt
from torch.utils.data import random_split, DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# === Step 1: Split Dataset (70% Train, 15% Val, 15% Test) ===
total_len = len(dataset)
train_len = int(0.7 * total_len)
val_len = int(0.15 * total_len)
test_len = total_len - train_len - val_len

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_len, val_len, test_len])

# === Step 2: Create DataLoaders ===
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)  # ✅ As per paper
val_loader = DataLoader(val_dataset, batch_size=2)

# === Step 3: Setup Optimizer, Scheduler, and Early Stopping ===
# Only update LoRA adapters + projector layer
trainable_params = list(filter(lambda p: p.requires_grad, list(model.parameters()) + list(projector.parameters())))
optimizer = AdamW(trainable_params, lr=5e-5)



num_epochs = 20
patience = 4
warmup_steps = 500
total_steps = len(train_loader) * num_epochs




scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=1000,
    num_training_steps=len(train_loader) * 20  # if num_epochs = 20
)

# Tracking
train_losses, val_losses = [], []
train_ppls, val_ppls = [], []
best_val_loss = float("inf")
epochs_no_improve = 0

# === Step 4: Training Loop ===
for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0
    start_time = time.time()

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        image_feature = batch["image_feature"].to(device)
        projected_feature = projector(image_feature)  # ✅ Apply projection
        encoder_hidden_states = projected_feature.unsqueeze(1)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            encoder_hidden_states=encoder_hidden_states,
            labels=input_ids
        )

        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # ✅ Gradient clipping
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        torch.cuda.empty_cache()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    train_ppls.append(math.exp(avg_train_loss))

    # === Step 5: Validation Loop ===
    model.eval()
    total_val_loss = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            image_feature = batch["image_feature"].to(device)
            projected_feature = projector(image_feature)
            encoder_hidden_states = projected_feature.unsqueeze(1)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                encoder_hidden_states=encoder_hidden_states,
                labels=input_ids
            )

            total_val_loss += outputs.loss.item()

    avg_val_loss = total_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    val_ppls.append(math.exp(avg_val_loss))

    # === Step 6: Early Stopping and Best Checkpoint Saving ===
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), "best_gpt2_model.pt")
        torch.save(projector.state_dict(), "best_projector.pt")
        print("📌 Best model saved.")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"⏹️ Early stopping triggered at epoch {epoch+1}")
            break

    # === Log Epoch Results ===
    print(f"✅ Epoch {epoch+1}/{num_epochs}")
    print(f"   Train Loss: {avg_train_loss:.4f}, Perplexity: {math.exp(avg_train_loss):.2f}")
    print(f"   Val   Loss: {avg_val_loss:.4f}, Perplexity: {math.exp(avg_val_loss):.2f}")
    print(f"   Time: {int(time.time() - start_time)}s\n")

# === Step 7: Plot Loss and Perplexity Curves ===
plt.figure(figsize=(12, 5))

# Loss
plt.subplot(1, 2, 1)
plt.plot(train_losses, label="Train Loss", marker='o')
plt.plot(val_losses, label="Val Loss", marker='o')
plt.title("Loss per Epoch")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

# Perplexity
plt.subplot(1, 2, 2)
plt.plot(train_ppls, label="Train Perplexity", marker='o')
plt.plot(val_ppls, label="Val Perplexity", marker='o')
plt.title("Perplexity per Epoch")
plt.xlabel("Epoch")
plt.ylabel("Perplexity")
plt.legend()

plt.tight_layout()
plt.show()





In [ ]:
# === Step 3.4: Inference — Generate Reports from Test Images ===

from transformers import GPT2Tokenizer
from torch.nn.functional import pad

# Set model to evaluation mode (disable dropout, fix layer norms)
model.eval()

results = []  # List to store results for all test images

# Loop over test dataset
for idx, sample in enumerate(test_dataset):
    # Prepare image features for cross-attention
    image_feature = sample["image_feature"].unsqueeze(0).to(device)  # add batch dimension — becomes [1, 768].
    encoder_hidden_states = image_feature.unsqueeze(1)  # add sequence length dimension(one image) — becomes [1, 1, 768]
                                                         #(what GPT-2 expects for cross-attention input).

    # Generate caption using the model
    generated = model.generate(
        input_ids=None,                          # Start from scratch (no text prompt), just condition on the image features.
        encoder_hidden_states=encoder_hidden_states,  # Image features for cross-attention
        max_length=70,                           # Maximum number of tokens to generate
        num_beams=3,                             # Beam search with 3 beams
        early_stopping=True,                     # Stop early if EOS is reached
        no_repeat_ngram_size=2,                  # Prevent repetition of 2-grams
        pad_token_id=tokenizer.pad_token_id      # f the sequence is shorter than max_length, as result of stop earlier, pad
    )

    # Decode generated token IDs into human-readable text.
    decoded_caption = tokenizer.decode(generated[0], skip_special_tokens=True) #skip_special_tokens=True
                                                                                #Removes padding, start/end tokens from the output.

    # Get ground-truth caption (remove padding tokens)
    true_caption = sample["input_ids"] #Take the input_ids (tokenized real caption).
    true_caption = true_caption[true_caption != tokenizer.pad_token_id] #Remove padded tokens (!= pad_token_id).
    reference = tokenizer.decode(true_caption, skip_special_tokens=True)#Decode the token IDs into readable text.

    # Save image path, reference caption, and generated caption
    results.append({
        "image_path": df.iloc[idx]["image_path"],        # Image path
        "reference_caption": df.iloc[idx]["caption"],    # Ground-truth caption from df
        "generated_caption": decoded_caption            # Model-generated caption
    })

# Save all results into a CSV file
results_df = pd.DataFrame(results)
results_df.to_csv("generated_reports.csv", index=False)
print("✅ Saved generated captions to 'generated_reports.csv'")





In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

# Load the saved results
df = pd.read_csv("generated_reports.csv")

# Show the first 10 samples
for idx in range(min(10, len(df))):
    image_path = df.iloc[idx]["image_path"]
    img = Image.open(image_path).convert("RGB")

    # Show the image
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"🩻 X-ray Image #{idx + 1}")
    plt.show()

    # Show the captions
    print(f"🟢 Ground Truth (Caption):\n{df.iloc[idx]['reference_caption']}\n")
    print(f"🔵 Generated Caption:\n{df.iloc[idx]['generated_caption']}\n")
    print("=" * 100)



In [ ]:
print("Pad token ID:", tokenizer.pad_token_id)
print("EOS token ID:", tokenizer.eos_token_id)


# Step 4: Evaluation of Generated Reports

1. **Load Results**:
   - Read `generated_reports.csv` containing generated and reference captions.

2. **Preprocess Captions**:
   - Tokenize captions for BLEU evaluation.
   - No special preprocessing for ROUGE-L or BERTScore.

3. **BLEU Evaluation**:
   - Compute BLEU-1, BLEU-2, BLEU-3, and BLEU-4 using `nltk` with smoothing.

4. **ROUGE-L Evaluation**:
   - Compute ROUGE-L F1 score using `rouge_scorer`.

5. **BERTScore Evaluation**:
   - Compute semantic similarity using `bert-score` package.
   - Model: `distilbert-base-uncased`.
   - Output: F1 score.

📊 **Final Metrics**:
- BLEU-1, BLEU-2, BLEU-3, BLEU-4
- ROUGE-L
- BERTScore (F1)

✅ **Outcome**:
> Quantitative evaluation of generated medical reports using both n-gram and semantic similarity metrics.




In [ ]:
!pip install rouge-score

In [ ]:
!pip install bert-score

In [ ]:
# === Step 4: Evaluation of Generated Reports ===

import pandas as pd
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from bert_score import score as bertscore
import nltk

# Download required NLTK resources for METEOR
nltk.download('wordnet')
nltk.download('omw-1.4')

# === Load Generated Captions and Reference Captions ===
# === Load Generated Captions and Reference Captions ===
results_df = pd.read_csv("generated_reports.csv")

# ✅ Fix: Ensure all captions are strings
results_df["generated_caption"] = results_df["generated_caption"].astype(str)
results_df["reference_caption"] = results_df["reference_caption"].astype(str)

generated_captions = results_df["generated_caption"].tolist()
reference_captions = results_df["reference_caption"].tolist()


# === Step 1: Preprocess for BLEU Evaluation ===
tokenized_preds = [pred.split() for pred in generated_captions]
tokenized_refs = [[ref.split()] for ref in reference_captions]  # nested list for corpus_bleu
smoothie = SmoothingFunction().method4  # smoothing for short sentences

# BLEU Scoring
bleu_1 = corpus_bleu(tokenized_refs, tokenized_preds, weights=(1, 0, 0, 0), smoothing_function=smoothie)
bleu_2 = corpus_bleu(tokenized_refs, tokenized_preds, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothie)
bleu_3 = corpus_bleu(tokenized_refs, tokenized_preds, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smoothie)
bleu_4 = corpus_bleu(tokenized_refs, tokenized_preds, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)

# === Step 2: ROUGE-L Evaluation ===
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
rouge_l_scores = [scorer.score(ref, pred)['rougeL'].fmeasure for ref, pred in zip(reference_captions, generated_captions)]
rouge_l = sum(rouge_l_scores) / len(rouge_l_scores)

# === Step 3: METEOR Evaluation ===
# === Step 3: METEOR Evaluation ===
meteor_scores = [
    meteor_score([ref.split()], pred.split())
    for ref, pred in zip(reference_captions, generated_captions)
]
meteor_avg = sum(meteor_scores) / len(meteor_scores)


# === Step 4: BERTScore Evaluation ===
P, R, F1 = bertscore(
    generated_captions,
    reference_captions,
    model_type="distilbert-base-uncased",
    lang='en',
    rescale_with_baseline=True,
    batch_size=4,

)
bertscore_f1 = F1.mean().item()

# === Final Result Printout ===
print("\n📊 Final Evaluation Results:")
print(f"BLEU-1: {bleu_1:.4f}")
print(f"BLEU-2: {bleu_2:.4f}")
print(f"BLEU-3: {bleu_3:.4f}")
print(f"BLEU-4: {bleu_4:.4f}")
print(f"ROUGE-L: {rouge_l:.4f}")
print(f"METEOR: {meteor_avg:.4f}")
print(f"BERTScore (F1): {bertscore_f1:.4f}")